# Downstream Application Task: Solar Flare

## Step 0: Task Preamble Set up

In [ ]:
#Step 1: Mounting Google Drive and Definign Stable Paths & Caches
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

import os, sys, pathlib

PROJECT_ROOT = "/content/drive/MyDrive/projects/surya"   # your workspace
REPO_DIR     = f"{PROJECT_ROOT}/repo"
SURYA_ROOT   = f"{REPO_DIR}/Surya"  # repo root (has surya/, tests/, downstream_examples/, etc.)

# outputs for THIS notebook
RUN_NAME = "solar_flare_infer_01"
OUT_DIR  = f"{PROJECT_ROOT}/out/{RUN_NAME}"
os.makedirs(OUT_DIR, exist_ok=True)

# persistent caches (prevents redownloading)
CACHE_ROOT = f"{PROJECT_ROOT}/colab_caches"
os.makedirs(CACHE_ROOT, exist_ok=True)
os.environ["HF_HOME"] = f"{CACHE_ROOT}/hf_home"
os.environ["XDG_CACHE_HOME"] = f"{CACHE_ROOT}/xdg_cache"
os.environ["TORCH_HOME"] = f"{CACHE_ROOT}/torch"
os.environ["TRANSFORMERS_CACHE"] = f"{CACHE_ROOT}/hf_home/transformers"

print("SURYA_ROOT:", SURYA_ROOT)
print("OUT_DIR:", OUT_DIR)
print("HF_HOME:", os.environ["HF_HOME"])

Mounted at /content/drive
SURYA_ROOT: /content/drive/MyDrive/projects/surya/repo/Surya
OUT_DIR: /content/drive/MyDrive/projects/surya/out/solar_flare_infer_01
HF_HOME: /content/drive/MyDrive/projects/surya/colab_caches/hf_home


In [ ]:
#Step2: Ensuring Surya Repo editable install
import os, subprocess, sys

assert os.path.exists(SURYA_ROOT), f"Surya repo not found at {SURYA_ROOT}"

%pip -q install -e "{SURYA_ROOT}" --no-deps

# Add SURYA_ROOT to sys.path to ensure 'surya' module is found
if SURYA_ROOT not in sys.path:
    sys.path.insert(0, SURYA_ROOT)

import surya
print("surya imported from:", surya.__file__)


  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for surya (pyproject.toml) ... done
surya imported from: /content/drive/MyDrive/projects/surya/repo/Surya/surya/__init__.py


In [ ]:
#Step3: Ensuring Environment Set Up
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):",
          torch.cuda.get_device_properties(0).total_memory / 1e9)


PyTorch: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA L4
GPU memory (GB): 23.65915136


# Step 1: Solar Flare Task Set Up

In [ ]:
#Step1: Locate Solar Flare Task Directory
import os, pathlib

SOLAR_FLARE_TASK_CANDIDATES = [
    f"{SURYA_ROOT}/downstream_examples/solar_flare_forcasting",
    f"{SURYA_ROOT}/downstream_examples/solar_flare_forecasting",
]

TASK_DIR = next((p for p in SOLAR_FLARE_TASK_CANDIDATES if os.path.exists(p)), None)
assert TASK_DIR, f"Solar flare task not found. Checked: {SOLAR_FLARE_TASK_CANDIDATES}"

print("✅ Solar flare task directory:")
print(TASK_DIR)


✅ Solar flare task directory:
/content/drive/MyDrive/projects/surya/repo/Surya/downstream_examples/solar_flare_forcasting


In [ ]:
# Step 2: Isolate Imports and Load Task Inference Code

import sys

# Ensure Surya repo is importable
if SURYA_ROOT not in sys.path:
    sys.path.insert(0, SURYA_ROOT)

# Ensure task dir is FIRST so infer.py resolves correctly
if TASK_DIR in sys.path:
    sys.path.remove(TASK_DIR)
sys.path.insert(0, TASK_DIR)

# Install missing deps Surya expects (safe/idempotent)
%pip -q install hdf5plugin sunpy mpl-animators



# Clear stale modules from previous downstream runs
for m in ("infer", "dataset", "finetune"):
    if m in sys.modules:
        del sys.modules[m]

from infer import (
    load_model,
    run_inference,
    get_dataloader,
    infer_single_sample,
)

print("✅ Solar flare infer.py imported successfully")


✅ Solar flare infer.py imported successfully


In [ ]:
#Step3: Check the Task Directory for Solar Flare Assests
%cd "{TASK_DIR}"
!ls -la assets || true
!test -f ./assets/solar_flare_weights.pth && echo "✅ weights present" || echo "❌ weights missing"

!test -f ./config_infer.yaml && echo "✅ config_infer.yaml present" || echo "config_infer.yaml missing"
!test -f ./config.yaml && echo "✅ config.yaml present" || echo "config.yaml missing"


/content/drive/MyDrive/projects/surya/repo/Surya/downstream_examples/solar_flare_forcasting
total 3529258
drwx------ 2 root root       4096 Jan 29 17:58 .cache
drwx------ 2 root root       4096 Jan 29 22:00 flare_capability_runs
-rw------- 1 root root          0 Jan 20 18:53 .gitkeep
drwx------ 2 root root       4096 Jan 29 17:59 infer_data
drwx------ 2 root root       4096 Jan 29 19:45 infer_data_live
drwx------ 2 root root       4096 Jan 29 21:43 infer_data_recent_jsoc
drwx------ 2 root root       4096 Jan 29 21:11 jsoc_raw_recent
-rw------- 1 root root       3835 Jan 29 18:01 scalers.yaml
-rw------- 1 root root 1794022584 Jan 29 17:58 solar_flare_weights.pth
-rw------- 1 root root 1800384309 Jan 29 17:59 surya.366m.v1.pt
drwx------ 2 root root       4096 Jan 29 17:58 surya-bench-flare-forecasting
-rw------- 1 root root   18557847 Jan 29 18:01 train_index_surya_1_0.csv
-rw------- 1 root root     961993 Jan 29 18:01 valid_index_surya_1_0.csv
✅ weights present
✅ config_infer.yaml prese

In [ ]:
# Step 4: Load configuration + checkpoint + base model (matches current disk layout)

import os, yaml, torch
from surya.utils.distributed import set_global_seed

%cd "{TASK_DIR}"

config_path = "./config_infer.yaml"
if not os.path.exists(config_path):
    config_path = "./config.yaml"
assert os.path.exists(config_path), "No config file found."

checkpoint_path = "./assets/solar_flare_weights.pth"
base_model_path = "./assets/surya.366m.v1.pt"

assert os.path.exists(checkpoint_path), f"Missing: {checkpoint_path}"
assert os.path.exists(base_model_path), f"Missing: {base_model_path}"

with open(config_path, "r") as f:
    config = yaml.safe_load(f)

set_global_seed(42)

dtype_map = {"float16": torch.float16, "bfloat16": torch.bfloat16, "float32": torch.float32}
config["dtype"] = dtype_map.get(config.get("dtype", "float32"), torch.float32)

print("✅ config:", config_path)
print("✅ checkpoint:", checkpoint_path)
print("✅ base model:", base_model_path)
print("✅ dtype:", config["dtype"])


/content/drive/MyDrive/projects/surya/repo/Surya/downstream_examples/solar_flare_forcasting
✅ config: ./config_infer.yaml
✅ checkpoint: ./assets/solar_flare_weights.pth
✅ base model: ./assets/surya.366m.v1.pt
✅ dtype: torch.float32


In [ ]:
# Step 5: Device Selection
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device_type = "cuda" if torch.cuda.is_available() else "cpu"
print("✅ device:", device)


✅ device: cuda


In [ ]:
# Step 6: Run baseline inference

import os
import yaml # Ensure yaml is imported here

RUN_NAME = "solar_flare_baseline"
OUT_DIR = f"{PROJECT_ROOT}/out/{RUN_NAME}"
os.makedirs(OUT_DIR, exist_ok=True)

data_type = "valid"   # try valid first
num_samples = 3

# Fix: Load scalers information into the config dictionary if not already present
if "scalers" not in config["data"] and "scalers_path" in config["data"]:
    # The config was loaded while TASK_DIR was the current working directory, so
    # config["data"]["scalers_path"] is relative to TASK_DIR.
    scalers_full_path = os.path.join(TASK_DIR, config["data"]["scalers_path"])
    with open(scalers_full_path, "r") as f:
        config["data"]["scalers"] = yaml.safe_load(f)
    print(f"✅ Loaded scalers from: {scalers_full_path}")
elif "scalers" in config["data"]:
    print("Scalers already present in config.")
else:
    print("Error: 'scalers_path' not found in config['data'] to load scalers.")


print("🔬 Running inference...")
run_inference(
    config=config,
    checkpoint_path=checkpoint_path,
    output_dir=OUT_DIR,
    device=device,
    data_type=data_type,
    num_samples=num_samples,
    device_type=device_type,
)

print("🎉 Done. Outputs in:", OUT_DIR)
!ls -la "{OUT_DIR}"


Scalers already present in config.
🔬 Running inference...
Loading model from ./assets/solar_flare_weights.pth
GPU is available
Loading pretrained model from ./assets/surya.366m.v1.pt.
Applying PEFT LoRA with configuration: {'r': 8, 'lora_alpha': 8, 'target_modules': ['q_proj', 'v_proj', 'k_proj', 'out_proj', 'fc1', 'fc2'], 'lora_dropout': 0.1, 'bias': 'none'}
trainable params: 1,024,000 || all params: 364,593,153 || trainable%: 0.28%
Dataset size: 3

Time Input           | Time Target          | Prediction      | Ground Truth
--------------------------------------------------------------------------------
2019-01-23T02:00     | 2019-01-23T04:00     | 0               | 0           

Time Input           | Time Target          | Prediction      | Ground Truth
--------------------------------------------------------------------------------
2019-01-23T02:00     | 2019-01-23T04:00     | 0               | 0           

Time Input           | Time Target          | Prediction      | Ground Tr

# Step 2: Solar Flare Task File Formats

In [ ]:
# Step 1: Print config expected dataset format
import pprint
print("=== config['data'] ===")
pprint.pprint(config.get("data", {}))
print("\n=== config['model'] ===")
pprint.pprint(config.get("model", {}))

# Inspect dataset code to see how it builds samples
import dataset, inspect
print(inspect.getsource(dataset))


=== config['data'] ===
{'batch_size': 1,
 'channels': ['aia94',
              'aia131',
              'aia171',
              'aia193',
              'aia211',
              'aia304',
              'aia335',
              'aia1600',
              'hmi_m',
              'hmi_bx',
              'hmi_by',
              'hmi_bz',
              'hmi_v'],
 'flare_data_path': './assets/surya-bench-flare-forecasting/validation.csv',
 'n_input_timestamps': 2,
 'num_data_workers': 4,
 'pooling': 1,
 'prefetch_factor': 2,
 'random_vert_flip': False,
 'scalers': {'aia131': {'base': 'surya.datasets.transformations',
                        'class': 'StandardScaler',
                        'count': 33571209216,
                        'epsilon': 1e-08,
                        'is_fitted': 'true',
                        'max': 4.642958164215088,
                        'mean': 0.03911230340600014,
                        'min': 0.0,
                        'sl_scale_factor': 0.01,
                 

In [ ]:
#Step2: Inspecting the exsiting Infer Index Schema
import pandas as pd
infer_idx_path = f"{TASK_DIR}/assets/infer_data/infer_index_surya_1_0.csv"
df = pd.read_csv(infer_idx_path)
print("infer_index columns:", list(df.columns))
display(df.head(5))
print("rows:", len(df))


infer_index columns: ['Unnamed: 0', 'path', 'timestep', 'present']


,Unnamed: 0,path,timestep,present
0,1,20110120_0100.nc,2011-01-20 01:00:00,1
1,2,20110120_0200.nc,2011-01-20 02:00:00,1
2,3,20110120_0300.nc,2011-01-20 03:00:00,1
3,4,20120115_0100.nc,2012-01-15 01:00:00,1
4,5,20120115_0200.nc,2012-01-15 02:00:00,1


rows: 15


In [ ]:
# Step3: Confirming what is inside the assets/infer_data file directory
import os, glob

infer_root = f"{TASK_DIR}/assets/infer_data"
print("infer_root:", infer_root)

files = sorted(glob.glob(infer_root + "/*.nc"))
print("nc files:", len(files))
for f in files:
    print(" -", os.path.basename(f))


infer_root: /content/drive/MyDrive/projects/surya/repo/Surya/downstream_examples/solar_flare_forcasting/assets/infer_data
nc files: 15
 - 20110120_0100.nc
 - 20110120_0200.nc
 - 20110120_0300.nc
 - 20120115_0048.nc
 - 20120115_0100.nc
 - 20120115_0148.nc
 - 20120115_0200.nc
 - 20120115_0248.nc
 - 20120115_0300.nc
 - 20130128_1812.nc
 - 20130128_1912.nc
 - 20130128_2012.nc
 - 20190123_0200.nc
 - 20190123_0300.nc
 - 20190123_0400.nc


In [ ]:
#Step 4:Inspect one of the .nc HDF5 Filter Files for variables and format required
%pip -q install h5py
%pip -q install hdf5plugin

import h5py
import hdf5plugin  # important: registers filters
import os

sample_nc = os.path.join(f"{TASK_DIR}/assets/infer_data", "20190123_0200.nc")
print("Opening with h5py:", sample_nc)

with h5py.File(sample_nc, "r") as f:
    print("Top-level keys:", list(f.keys()))
    # show a few groups/datasets and their shapes
    def walk(name, obj):
        if isinstance(obj, h5py.Dataset):
            print("DATASET:", name, obj.shape, obj.dtype)
    f.visititems(walk)


Opening with h5py: /content/drive/MyDrive/projects/surya/repo/Surya/downstream_examples/solar_flare_forcasting/assets/infer_data/20190123_0200.nc
Top-level keys: ['y', 'x', 'aia94', 'aia131', 'aia171', 'aia193', 'aia211', 'aia304', 'aia335', 'aia1600', 'hmi_m', 'hmi_bx', 'hmi_by', 'hmi_bz', 'hmi_v']
DATASET: aia131 (4096, 4096) float32
DATASET: aia1600 (4096, 4096) float32
DATASET: aia171 (4096, 4096) float32
DATASET: aia193 (4096, 4096) float32
DATASET: aia211 (4096, 4096) float32
DATASET: aia304 (4096, 4096) float32
DATASET: aia335 (4096, 4096) float32
DATASET: aia94 (4096, 4096) float32
DATASET: hmi_bx (4096, 4096) float32
DATASET: hmi_by (4096, 4096) float32
DATASET: hmi_bz (4096, 4096) float32
DATASET: hmi_m (4096, 4096) float32
DATASET: hmi_v (4096, 4096) float32
DATASET: x (4096,) >f4
DATASET: y (4096,) >f4


In [ ]:
#Step5: Inspect the x and y content datasets
import h5py, hdf5plugin, numpy as np, os

sample_nc = os.path.join(f"{TASK_DIR}/assets/infer_data", "20190123_0200.nc")
with h5py.File(sample_nc, "r") as f:
    x = f["x"][:]
    y = f["y"][:]
    print("x:", x.shape, x.dtype, "min/max:", float(x.min()), float(x.max()))
    print("y:", y.shape, y.dtype, "min/max:", float(y.min()), float(y.max()))
    print("x head:", x[:5])
    print("y head:", y[:5])


x: (4096,) >f4 min/max: 0.0 0.0
y: (4096,) >f4 min/max: 0.0 0.0
x head: [0. 0. 0. 0. 0.]
y head: [0. 0. 0. 0. 0.]


In [ ]:
#Step 6: Testing Custom .nc CSV File
import os
LIVE_ROOT = f"{TASK_DIR}/assets/infer_data_live"
os.makedirs(LIVE_ROOT, exist_ok=True)
print("✅ LIVE_ROOT:", LIVE_ROOT)

import os, shutil, pandas as pd

# Define the relevant timestamps for a valid sample
timestamps_to_include = [
    "2019-01-23 02:00:00",
    "2019-01-23 03:00:00", # This will be the reference timestamp for the inference
    "2019-01-23 04:00:00",
]

# 1) Copy known-good nc files into LIVE_ROOT
for ts_str in timestamps_to_include:
    # Convert 'YYYY-MM-DD HH:MM:SS' to 'YYYYMMDD_HHMM.nc'
    src_name = pd.to_datetime(ts_str).strftime("%Y%m%d_%H%M.nc")
    src = os.path.join(f"{TASK_DIR}/assets/infer_data", src_name)
    dst = os.path.join(LIVE_ROOT, src_name)
    shutil.copy2(src, dst)
    print(f"✅ copied {src_name} to {LIVE_ROOT}")

# 2) Write a minimal infer index for all required timestamps
live_index_data = []
for ts_str in timestamps_to_include:
    src_name = pd.to_datetime(ts_str).strftime("%Y%m%d_%H%M.nc")
    live_index_data.append({
        "path": src_name,
        "timestep": ts_str,
        "present": 1
    })
live_index = pd.DataFrame(live_index_data)
live_index_path = os.path.join(LIVE_ROOT, "infer_index_live.csv")
live_index.to_csv(live_index_path, index=False)

print("✅ wrote:", live_index_path)
display(live_index)

# 2.1) Create a minimal flare index for the live samples
live_flare_index_data = []
for ts_str in timestamps_to_include:
    live_flare_index_data.append({
        "timestamp": ts_str,
        "max_goes_class": "B0.0", # dummy value
        "cumulative_index": 0.0,
        "label_max": 0,
        "label_cum": 0,
    })
live_flare_index = pd.DataFrame(live_flare_index_data)
live_flare_index_path = os.path.join(LIVE_ROOT, "live_flare_index.csv")
live_flare_index.to_csv(live_flare_index_path, index=False)
print("✅ wrote:", live_flare_index_path)
display(live_flare_index)


# 3) Point config to LIVE_ROOT and the new indices
config["data"]["sdo_data_root_path"] = LIVE_ROOT
config["data"]["valid_data_path"] = live_index_path
config["data"]["flare_data_path"] = live_flare_index_path # Point to the new flare index

# 4) Run inference for 1 sample (using 2019-01-23 03:00:00 as the reference)
RUN_NAME = "solar_flare_live_index_test"
OUT_DIR2 = f"{PROJECT_ROOT}/out/{RUN_NAME}"
os.makedirs(OUT_DIR2, exist_ok=True)

run_inference(
    config=config,
    checkpoint_path=checkpoint_path,
    output_dir=OUT_DIR2,
    device=device,
    data_type="valid",
    num_samples=1, # Request only one sample for testing the valid flow
    device_type="cuda" if str(device)=="cuda" else "cpu",
)

print("🎉 Live-index test complete.")


✅ LIVE_ROOT: /content/drive/MyDrive/projects/surya/repo/Surya/downstream_examples/solar_flare_forcasting/assets/infer_data_live
✅ copied 20190123_0200.nc to /content/drive/MyDrive/projects/surya/repo/Surya/downstream_examples/solar_flare_forcasting/assets/infer_data_live
✅ copied 20190123_0300.nc to /content/drive/MyDrive/projects/surya/repo/Surya/downstream_examples/solar_flare_forcasting/assets/infer_data_live
✅ copied 20190123_0400.nc to /content/drive/MyDrive/projects/surya/repo/Surya/downstream_examples/solar_flare_forcasting/assets/infer_data_live
✅ wrote: /content/drive/MyDrive/projects/surya/repo/Surya/downstream_examples/solar_flare_forcasting/assets/infer_data_live/infer_index_live.csv


,path,timestep,present
0,20190123_0200.nc,2019-01-23 02:00:00,1
1,20190123_0300.nc,2019-01-23 03:00:00,1
2,20190123_0400.nc,2019-01-23 04:00:00,1


✅ wrote: /content/drive/MyDrive/projects/surya/repo/Surya/downstream_examples/solar_flare_forcasting/assets/infer_data_live/live_flare_index.csv


,timestamp,max_goes_class,cumulative_index,label_max,label_cum
0,2019-01-23 02:00:00,B0.0,0.0,0,0
1,2019-01-23 03:00:00,B0.0,0.0,0,0
2,2019-01-23 04:00:00,B0.0,0.0,0,0


Loading model from ./assets/solar_flare_weights.pth
GPU is available
Loading pretrained model from ./assets/surya.366m.v1.pt.
Applying PEFT LoRA with configuration: {'r': 8, 'lora_alpha': 8, 'target_modules': ['q_proj', 'v_proj', 'k_proj', 'out_proj', 'fc1', 'fc2'], 'lora_dropout': 0.1, 'bias': 'none'}
trainable params: 1,024,000 || all params: 364,593,153 || trainable%: 0.28%
Dataset size: 1

Time Input           | Time Target          | Prediction      | Ground Truth
--------------------------------------------------------------------------------
2019-01-23T02:00     | 2019-01-23T04:00     | 0               | 0           
🎉 Live-index test complete.


# Step 3: Solar Flare Outputs

Attempt at creating a pipeline for producing custom solar flare predictions given any defined time range. This time range is used for querying JSOC servers for SDO data through SunPy Library.

In [ ]:
# Step 1: imports
%pip install -q -U drms "sunpy[all]" astropy zeep reproject h5py hdf5plugin netCDF4 xarray tqdm hf-transfer>=0.1.9

import os
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import astropy.units as u
from astropy.time import Time, TimeDelta
import drms
from sunpy.net import Fido, attrs as a
from astropy.io import fits
from tqdm.auto import tqdm



In [ ]:
# Step 2: Config: paths, channels, JSOC email

JSOC_EMAIL = "kylienager330@gmail.com"
client = drms.Client(email=JSOC_EMAIL)

OUT_ROOT = Path("./assets/infer_data")
RAW_DIR  = OUT_ROOT / "raw"
NC_DIR   = OUT_ROOT / "nc"
RAW_DIR.mkdir(parents=True, exist_ok=True)
NC_DIR.mkdir(parents=True, exist_ok=True)

CHANNELS = [
    "aia94","aia131","aia171","aia193","aia211","aia304","aia335","aia1600",
    "hmi_bx","hmi_by","hmi_bz","hmi_m","hmi_v"
]

for ch in CHANNELS:
    (RAW_DIR / ch).mkdir(parents=True, exist_ok=True)

print("RAW_DIR:", RAW_DIR)
print("NC_DIR :", NC_DIR)
print("Example channel dir exists:", (RAW_DIR/"aia193").exists())

RAW_DIR: assets/infer_data/raw
NC_DIR : assets/infer_data/nc
Example channel dir exists: True


In [ ]:
#Step 3: List cadidate series and check their segments
# Uses drms (you already created `client = drms.Client(email=JSOC_EMAIL)`)

def show_series_info(series):
    print("\n=== ", series, "===")
    try:
        info = client.info(series)
        print("Prime keys:", info.primekeys)
        print("Segments :", list(info.segments.index))
    except Exception as e:
        print("ERROR:", e)

# Common candidates (don’t assume these are right; we’ll verify)
for s in ["aia.lev1_euv_12s", "hmi.B_720s", "hmi.M_720s", "hmi.V_720s",
          "hmi.B_720s_nrt", "hmi.M_720s_nrt", "hmi.V_720s_nrt"]:
    show_series_info(s)



===  aia.lev1_euv_12s ===
Prime keys: ['T_REC', 'WAVELNTH']
Segments : ['image', 'spikes']

===  hmi.B_720s ===
Prime keys: ['T_REC']
Segments : ['inclination', 'azimuth', 'disambig', 'field', 'vlos_mag', 'dop_width', 'eta_0', 'damping', 'src_continuum', 'src_grad', 'alpha_mag', 'chisq', 'conv_flag', 'inclination_err', 'azimuth_err', 'field_err', 'vlos_err', 'alpha_err', 'field_inclination_err', 'field_az_err', 'inclin_azimuth_err', 'field_alpha_err', 'inclination_alpha_err', 'azimuth_alpha_err', 'conf_disambig', 'info_map', 'confid_map']

===  hmi.M_720s ===
Prime keys: ['T_REC', 'CAMERA']
Segments : ['magnetogram']

===  hmi.V_720s ===
Prime keys: ['T_REC', 'CAMERA']
Segments : ['Dopplergram']

===  hmi.B_720s_nrt ===
ERROR: Series hmi.b_720s_nrt is not a valid series accessible from hmidb2. [status=5]

===  hmi.M_720s_nrt ===
Prime keys: ['T_REC', 'CAMERA']
Segments : ['magnetogram']

===  hmi.V_720s_nrt ===
ERROR: Series hmi.v_720s_nrt is not a valid series accessible from hmidb2.

In [ ]:
#Step 4: Defining and identifying channels
import astropy.units as u
from sunpy.net import Fido, attrs as a

# AIA
AIA_SERIES = "aia.lev1_euv_12s"
AIA_SEGMENT = "image"
AIA_WAVE = {
    "aia94": 94, "aia131": 131, "aia171": 171, "aia193": 193,
    "aia211": 211, "aia304": 304, "aia335": 335, "aia1600": 1600
}

# HMI
HMI_VEC_SERIES = "hmi.B_720s"     # has: field, inclination, azimuth
HMI_MAG_SERIES = "hmi.M_720s"     # magnetogram
HMI_DOP_SERIES = "hmi.V_720s"     # Dopplergram

HMI_VEC_SEGS = {"field": "field", "inclination": "inclination", "azimuth": "azimuth"}
HMI_MAG_SEG  = "magnetogram"
HMI_DOP_SEG  = "Dopplergram"


In [ ]:
#Step 5: JSOC search functions
import astropy.units as u
from sunpy.net import Fido, attrs as a

# AIA series: EUV vs UV
AIA_EUV_SERIES = "aia.lev1_euv_12s"
AIA_UV_SERIES  = "aia.lev1_uv_24s"
AIA_SEGMENT    = "image"

def jsoc_search_aia(channel, start, end, cadence=1*u.min):
    """
    AIA EUV channels (94-335) are in aia.lev1_euv_12s.
    AIA 1600 is UV and must use aia.lev1_uv_24s.
    """
    wave = AIA_WAVE[channel] * u.AA
    series = AIA_UV_SERIES if channel == "aia1600" else AIA_EUV_SERIES

    return Fido.search(
        a.Time(start, end),
        a.jsoc.Series(series),
        a.jsoc.Segment(AIA_SEGMENT),
        a.Wavelength(wave),
        a.Sample(cadence),
        a.jsoc.Notify(JSOC_EMAIL),
    )

def jsoc_search_hmi(series, segment, start, end, cadence=12*u.min, camera=None):
    """
    HMI M_720s and V_720s may require CAMERA prime key.
    We support optional camera constraint.
    """
    req = [
        a.Time(start, end),
        a.jsoc.Series(series),
        a.jsoc.Segment(segment),
        a.Sample(cadence),
        a.jsoc.Notify(JSOC_EMAIL),
    ]
    if camera is not None:
        req.append(a.jsoc.PrimeKey("CAMERA", camera))
    return Fido.search(*req)

def jsoc_fetch(resp, out_dir: Path):
    out_dir.mkdir(parents=True, exist_ok=True)
    files = Fido.fetch(resp, path=str(out_dir))
    return list(files)


In [ ]:
# Step 6: Auto-detect CAMERA for HMI M/V products

import astropy.units as u
from astropy.time import Time

# Define START and END for the search
# These should define the time range you are interested in for fetching HMI data

START = Time("2014-03-29 15:45:00")
END = Time("2014-03-29 19:05:00")

def find_working_camera(series, segment, start, end):
    for cam in (1, 2):
        resp = jsoc_search_hmi(series, segment, start, end, cadence=12*u.min, camera=cam)
        if len(resp) > 0:
            return cam
    return None

MAG_CAMERA = find_working_camera(HMI_MAG_SERIES, HMI_MAG_SEG, START, END)
DOP_CAMERA = find_working_camera(HMI_DOP_SERIES, HMI_DOP_SEG, START, END)

print("MAG_CAMERA:", MAG_CAMERA)
print("DOP_CAMERA:", DOP_CAMERA)


MAG_CAMERA: 1
DOP_CAMERA: 1


In [ ]:
# Step 7: Download all required channels

#All AIA channels are as listed. For google co labs sake, it is too data heavy running on all at once.
#AIA_CHANNELS = ["aia94","aia131","aia171","aia193","aia211","aia304","aia335","aia1600"]

#using fewer channels temporarily to continue on building up this notebook!
AIA_CHANNELS = ["aia131","aia171","aia193","aia304"]

import os

AIA_CADENCE = 2*u.min
HMI_CADENCE = 12*u.min

def already_downloaded(folder, min_files=10):
    """
    True if folder exists and has at least `min_files` .fits files.
    Prevents re-downloading after Colab restarts.
    """
    if not os.path.isdir(folder):
        return False
    count = 0
    for f in os.listdir(folder):
        if f.lower().endswith(".fits"):
            count += 1
            if count >= min_files:
                return True
    return False

def list_fits(folder):
    """Return sorted list of FITS file paths in folder (empty if missing)."""
    if not os.path.isdir(folder):
        return []
    return [os.path.join(folder, f) for f in sorted(os.listdir(folder)) if f.lower().endswith(".fits")]

def ensure_dir(folder):
    os.makedirs(folder, exist_ok=True)

def download_all_required(start, end):
    files = {}

    # AIA (sampling)
    for ch in AIA_CHANNELS:
        out_dir = RAW_DIR/ch
        ensure_dir(out_dir)

        # skip if already downloaded
        if already_downloaded(out_dir, min_files=15):
            files[ch] = list_fits(out_dir)
            print(ch, "already downloaded, skipping. found:", len(files[ch]))
            continue

        resp = jsoc_search_aia(ch, start, end, cadence=AIA_CADENCE)
        files[ch] = jsoc_fetch(resp, out_dir)
        print(ch, "downloaded:", len(files[ch]))

    # HMI vectors (bx/by/bz computed later)
    for seg in ("field", "inclination", "azimuth"):
        key = f"hmi_{seg}"
        out_dir = RAW_DIR/key
        ensure_dir(out_dir)

        # skip if already downloaded
        if already_downloaded(out_dir, min_files=3):
            files[key] = list_fits(out_dir)
            print(key, "already downloaded, skipping. found:", len(files[key]))
            continue

        resp = jsoc_search_hmi(HMI_VEC_SERIES, seg, start, end, cadence=HMI_CADENCE)
        files[key] = jsoc_fetch(resp, out_dir)
        print(key, "downloaded:", len(files[key]))

    # HMI magnetogram (use detected camera if found, else no camera constraint)
    out_dir = RAW_DIR/"hmi_m"
    ensure_dir(out_dir)

    if already_downloaded(out_dir, min_files=3):
        files["hmi_m"] = list_fits(out_dir)
        print("hmi_m already downloaded, skipping. found:", len(files["hmi_m"]))
    else:
        resp = jsoc_search_hmi(HMI_MAG_SERIES, HMI_MAG_SEG, start, end, cadence=HMI_CADENCE, camera=MAG_CAMERA)
        if len(resp) == 0 and MAG_CAMERA is not None:
            resp = jsoc_search_hmi(HMI_MAG_SERIES, HMI_MAG_SEG, start, end, cadence=HMI_CADENCE, camera=None)
        files["hmi_m"] = jsoc_fetch(resp, out_dir)
        print("hmi_m downloaded:", len(files["hmi_m"]))

    # HMI Dopplergram (use detected camera if found, else no camera constraint)
    out_dir = RAW_DIR/"hmi_v"
    ensure_dir(out_dir)

    if already_downloaded(out_dir, min_files=3):
        files["hmi_v"] = list_fits(out_dir)
        print("hmi_v already downloaded, skipping. found:", len(files["hmi_v"]))
    else:
        resp = jsoc_search_hmi(HMI_DOP_SERIES, HMI_DOP_SEG, start, end, cadence=HMI_CADENCE, camera=DOP_CAMERA)
        if len(resp) == 0 and DOP_CAMERA is not None:
            resp = jsoc_search_hmi(HMI_DOP_SERIES, HMI_DOP_SEG, start, end, cadence=HMI_CADENCE, camera=None)
        files["hmi_v"] = jsoc_fetch(resp, out_dir)
        print("hmi_v downloaded:", len(files["hmi_v"]))

    return files

files = download_all_required(START, END)
print("File counts:", {k: len(v) for k, v in files.items()})

aia131 already downloaded, skipping. found: 41
aia171 already downloaded, skipping. found: 41
aia193 already downloaded, skipping. found: 41
aia304 already downloaded, skipping. found: 41
hmi_field already downloaded, skipping. found: 7
hmi_inclination already downloaded, skipping. found: 7
hmi_azimuth already downloaded, skipping. found: 7
hmi_m already downloaded, skipping. found: 7
hmi_v already downloaded, skipping. found: 7
File counts: {'aia131': 41, 'aia171': 41, 'aia193': 41, 'aia304': 41, 'hmi_field': 7, 'hmi_inclination': 7, 'hmi_azimuth': 7, 'hmi_m': 7, 'hmi_v': 7}


In [ ]:
# Step 8: Build indices + build Surya samples + write NetCDF + infer_index_custom.csv

import numpy as np
import pandas as pd
import xarray as xr
import sunpy.map
from astropy.io import fits
from astropy.time import Time, TimeDelta
from tqdm.auto import tqdm
import shutil
import tempfile
from pathlib import Path

# Matches your downloaded setup
TIME_DELTAS_INPUT_MIN = [-60, 0]   # <-- CORRECT: past context to now
ANCHOR = "aia193"

# Surya file-format expectation
REQUIRED_SHAPE = (4096, 4096)

# Channels we will write into each .nc (match your downloads + derived HMI products)
CHANNELS = AIA_CHANNELS + ["hmi_bx", "hmi_by", "hmi_bz", "hmi_m", "hmi_v"]

def obs_time_from_fits(fp):
    for ext in (1, 0):
        try:
            hdr = fits.getheader(fp, ext)
        except Exception:
            continue
        for key in ("DATE-OBS", "DATE_OBS", "T_OBS", "T_REC", "DATE"):
            if key in hdr and hdr[key]:
                val = hdr[key]
                if isinstance(val, bytes):
                    val = val.decode("utf-8", errors="ignore")
                return str(val).strip()
    return None

def build_file_index(file_list, label=""):
    if not file_list:
        print(f"[WARN] {label}: 0 files")
        return pd.DataFrame(columns=["time", "path", "time_unix"])

    rows = []
    bad = 0
    for fp in file_list:
        t = obs_time_from_fits(fp)
        if t is None:
            bad += 1
            continue
        rows.append((t, str(fp)))

    if not rows:
        print(f"[WARN] {label}: no parsable times (files={len(file_list)}, bad={bad})")
        return pd.DataFrame(columns=["time", "path", "time_unix"])

    df = pd.DataFrame(rows, columns=["time", "path"])

    unix = []
    keep = []
    for t in df["time"].astype(str).values:
        try:
            unix.append(Time(t).unix)
            keep.append(True)
        except Exception:
            unix.append(np.nan)
            keep.append(False)

    df["time_unix"] = unix
    df = df[keep].sort_values("time_unix").reset_index(drop=True)

    if bad > 0:
        print(f"[INFO] {label}: skipped {bad}/{len(file_list)} missing-time FITS")

    return df

def nearest_path(df_index, target_isot, max_offset_sec):
    if df_index is None or len(df_index) == 0:
        return None
    target = Time(target_isot).unix
    arr = df_index["time_unix"].values
    i = int(np.argmin(np.abs(arr - target)))
    best = df_index.iloc[i]
    if abs(best["time_unix"] - target) > max_offset_sec:
        return None
    return best["path"]

def resize_to_required(arr, target_shape=REQUIRED_SHAPE):
    ty, tx = target_shape
    y, x = arr.shape

    # center crop
    if y > ty:
        y0 = (y - ty) // 2
        arr = arr[y0:y0+ty, :]
    if x > tx:
        x0 = (x - tx) // 2
        arr = arr[:, x0:x0+tx]

    # pad
    y, x = arr.shape
    if y < ty or x < tx:
        out = np.zeros((ty, tx), dtype=arr.dtype)
        y0 = (ty - y) // 2
        x0 = (tx - x) // 2
        out[y0:y0+y, x0:x0+x] = arr
        arr = out

    return arr

def fits_to_array(fp):
    m = sunpy.map.Map(fp)
    data = np.array(m.data, dtype=np.float32)
    data = np.nan_to_num(data, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
    return resize_to_required(data)

def hmi_to_bxyz(field_arr, incl_deg_arr, az_deg_arr):
    incl = np.deg2rad(incl_deg_arr)
    az   = np.deg2rad(az_deg_arr)
    B = field_arr
    Bz = B * np.cos(incl)
    Bx = B * np.sin(incl) * np.cos(az)
    By = B * np.sin(incl) * np.sin(az)
    return Bx.astype(np.float32), By.astype(np.float32), Bz.astype(np.float32)

def write_surya_netcdf(reference_time_isot, arrays_by_channel_and_offset):
    ref_dt = Time(reference_time_isot).datetime
    NC_DIR.mkdir(parents=True, exist_ok=True)

    # Unique suffix to avoid collisions across reruns
    stamp = int(Time(reference_time_isot).unix * 1000)

    with tempfile.TemporaryDirectory() as tmpdir:
        tmp_path = Path(tmpdir) / f"infer_{ref_dt:%Y%m%d_%H%M%S}_{stamp}.nc"

        offsets = sorted(TIME_DELTAS_INPUT_MIN)

        x = np.zeros((REQUIRED_SHAPE[1],), dtype=np.float32)
        y = np.zeros((REQUIRED_SHAPE[0],), dtype=np.float32)

        ds = xr.Dataset(
            data_vars={
                ch: (("t","y","x"), np.stack([arrays_by_channel_and_offset[ch][off] for off in offsets], axis=0))
                for ch in CHANNELS
            },
            coords={
                "t": np.array(offsets, dtype=np.int32),
                "x": x,
                "y": y,
            },
            attrs={"reference_timestamp": reference_time_isot},
        )

        ds.to_netcdf(tmp_path, engine="h5netcdf")

        final_path = NC_DIR / f"infer_{ref_dt:%Y%m%d_%H%M%S}_{stamp}.nc"
        shutil.move(str(tmp_path), str(final_path))
        return str(final_path)

# Build indices dict ONCE
indices = {k: build_file_index(v, label=k) for k, v in files.items()}
print("Index sizes:", {k: len(v) for k, v in indices.items()})

def build_surya_samples(indices, start, end, cadence_sec=120, max_offset_sec=10*60):
    times = pd.date_range(Time(start).datetime, Time(end).datetime, freq=f"{cadence_sec}S")

    rows = []
    skipped = 0

    for center_dt in tqdm(times, desc="Building Surya samples"):
        center_isot = Time(center_dt).isot

        anchor_fp = nearest_path(indices[ANCHOR], center_isot, max_offset_sec=max_offset_sec)
        if anchor_fp is None:
            skipped += 1
            continue

        ref_time = obs_time_from_fits(anchor_fp)
        if ref_time is None:
            skipped += 1
            continue

        arrays = {ch: {} for ch in CHANNELS}

        for off in TIME_DELTAS_INPUT_MIN:
            desired = (Time(ref_time) + TimeDelta(off*60, format="sec")).isot

            # AIA
            for ch in AIA_CHANNELS:
                fp = nearest_path(indices[ch], desired, max_offset_sec=max_offset_sec)
                if fp is None:
                    arrays = None
                    break
                arrays[ch][off] = fits_to_array(fp)
            if arrays is None:
                break

            # HMI vector -> bx/by/bz
            pf = nearest_path(indices["hmi_field"], desired, max_offset_sec=max_offset_sec)
            pi = nearest_path(indices["hmi_inclination"], desired, max_offset_sec=max_offset_sec)
            pa = nearest_path(indices["hmi_azimuth"], desired, max_offset_sec=max_offset_sec)
            if None in (pf, pi, pa):
                arrays = None
                break

            B   = fits_to_array(pf)
            inc = fits_to_array(pi)
            az  = fits_to_array(pa)
            bx, by, bz = hmi_to_bxyz(B, inc, az)

            arrays["hmi_bx"][off] = resize_to_required(bx)
            arrays["hmi_by"][off] = resize_to_required(by)
            arrays["hmi_bz"][off] = resize_to_required(bz)

            # HMI m & v
            pm = nearest_path(indices["hmi_m"], desired, max_offset_sec=max_offset_sec)
            pv = nearest_path(indices["hmi_v"], desired, max_offset_sec=max_offset_sec)
            if None in (pm, pv):
                arrays = None
                break

            arrays["hmi_m"][off] = fits_to_array(pm)
            arrays["hmi_v"][off] = fits_to_array(pv)

        if arrays is None:
            skipped += 1
            continue

        # sanity check once
        if len(rows) == 0:
            for ch in CHANNELS:
                for off in TIME_DELTAS_INPUT_MIN:
                    print(ch, off, arrays[ch][off].shape, arrays[ch][off].dtype)

        nc_path = write_surya_netcdf(ref_time, arrays)
        rows.append({"timestamp": ref_time, "file_path": nc_path})

    df = pd.DataFrame(rows)
    out_csv = OUT_ROOT / "infer_index_custom.csv"
    df.to_csv(out_csv, index=False)
    print("Wrote:", out_csv, "| rows:", len(df), "| skipped:", skipped)
    return df

df = build_surya_samples(indices, START, END, cadence_sec=120, max_offset_sec=10*60)
print("df rows:", len(df))
df.head()

Index sizes: {'aia131': 41, 'aia171': 41, 'aia193': 41, 'aia304': 41, 'hmi_field': 7, 'hmi_inclination': 7, 'hmi_azimuth': 7, 'hmi_m': 7, 'hmi_v': 7}


Building Surya samples:   0%|          | 0/101 [00:00<?, ?it/s]

aia131 -60 (4096, 4096) float32
aia131 0 (4096, 4096) float32
aia171 -60 (4096, 4096) float32
aia171 0 (4096, 4096) float32
aia193 -60 (4096, 4096) float32
aia193 0 (4096, 4096) float32
aia304 -60 (4096, 4096) float32
aia304 0 (4096, 4096) float32
hmi_bx -60 (4096, 4096) float32
hmi_bx 0 (4096, 4096) float32
hmi_by -60 (4096, 4096) float32
hmi_by 0 (4096, 4096) float32
hmi_bz -60 (4096, 4096) float32
hmi_bz 0 (4096, 4096) float32
hmi_m -60 (4096, 4096) float32
hmi_m 0 (4096, 4096) float32
hmi_v -60 (4096, 4096) float32
hmi_v 0 (4096, 4096) float32
Wrote: assets/infer_data/infer_index_custom.csv | rows: 20 | skipped: 81
df rows: 20


,timestamp,file_path
0,2014-03-29T17:37:06.837,assets/infer_data/nc/infer_20140329_173706_139...
1,2014-03-29T17:39:06.837,assets/infer_data/nc/infer_20140329_173906_139...
2,2014-03-29T17:41:06.837,assets/infer_data/nc/infer_20140329_174106_139...
3,2014-03-29T17:43:06.837,assets/infer_data/nc/infer_20140329_174306_139...
4,2014-03-29T17:45:06.837,assets/infer_data/nc/infer_20140329_174506_139...


In [ ]:
#Step 9: Ensuring Correct data set formats and directory locations
from pathlib import Path

OUT_ROOT = Path("assets/infer_data")
infer_index_path = OUT_ROOT / "infer_index_custom.csv"
flare_index_path = OUT_ROOT / "flare_index_custom.csv"

# Most common Surya config layout
config["data"]["sdo_data_root_path"] = str(OUT_ROOT)
config["data"]["valid_data_path"] = str(infer_index_path)
config["data"]["flare_data_path"] = str(flare_index_path)

print("sdo_data_root_path:", config["data"]["sdo_data_root_path"])
print("valid_data_path:", config["data"]["valid_data_path"])
print("flare_data_path:", config["data"]["flare_data_path"])

import pandas as pd, xarray as xr, os

df = pd.read_csv("assets/infer_data/infer_index_custom.csv")
print("infer rows:", len(df))
first_nc = df["file_path"].iloc[0]
print("First nc exists:", os.path.exists(first_nc), first_nc)

ds = xr.open_dataset(first_nc, engine="h5netcdf")
print("Vars:", list(ds.data_vars.keys()))
print("Dims:", ds.dims)
print("t:", ds["t"].values)

sdo_data_root_path: assets/infer_data
valid_data_path: assets/infer_data/infer_index_custom.csv
flare_data_path: assets/infer_data/flare_index_custom.csv


In [1]:
#Step 10: Inference Run

import pandas as pd

flare_csv = "assets/infer_data/flare_index_custom.csv"
df = pd.read_csv(flare_csv)

# Explicitly remove 'present' and 'timestep' columns if they exist
if "present" in df.columns:
    df = df.drop(columns=["present"])
if "timestep" in df.columns:
    df = df.drop(columns=["timestep"])

df.to_csv(flare_csv, index=False)
print("flare CSV columns:", list(df.columns))
df.head()
print("=== START INFERENCE ===")
print("OUT_DIR:", OUT_DIR)
print("num_samples:", num_samples)

run_inference(
    config=config,
    checkpoint_path=checkpoint_path,
    output_dir=OUT_DIR,
    device=DEVICE,
    device_type=device_type,
    data_type="valid",
    num_samples=num_samples,
)

print("=== INFERENCE FINISHED ===")
!ls -lah "{OUT_DIR}"


FileNotFoundError: [Errno 2] No such file or directory: 'assets/infer_data/flare_index_custom.csv'

In [ ]:
#Step 11: Diplay Inference Predciton Results
import glob, os, pandas as pd
import matplotlib.pyplot as plt

csvs = sorted(glob.glob(os.path.join(OUT_DIR, "*.csv")), key=os.path.getmtime)
print("CSV outputs found:", csvs)

pred_path = csvs[-1]
preds = pd.read_csv(pred_path)

print("Prediction columns:", list(preds.columns))
display(preds.head(20))

# pick columns
tcol = "timestamp" if "timestamp" in preds.columns else [c for c in preds.columns if "time" in c.lower()][0]
ycol = [c for c in preds.columns if any(k in c.lower() for k in ["prob","pred","score","logit"])][0]

preds[tcol] = pd.to_datetime(preds[tcol], errors="coerce")
preds = preds.dropna(subset=[tcol]).sort_values(tcol)

plt.figure()
plt.plot(preds[tcol], preds[ycol])
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

CSV outputs found: []


IndexError: list index out of range